# Notebook 05: FastAPI Backend

Wraps the shared ScamSense pipeline into a REST API and logs every prediction to Supabase (PostgreSQL).

| Cell | What happens |
|---|---|
| 1  | Install dependencies |
| 1a | Load the shared pipeline module (`scamsense_pipeline.py`) |
| 2  | Load secrets + configure paths |
| 3  | Init shared pipeline: load model (HF) + build/cache FAISS index |
| 4  | Supabase: connect + verify the `predictions` table |
| 5  | Presentation helpers (explanation formatting) + Supabase logging |
| 6  | FastAPI app — /health, /predict, /stats, /history |
| 7  | Run server + test all endpoints with httpx |
| 8  | Export prediction log to CSV |
| 9  | Telegram bot (ScamSense Scout) |

**Inputs**: `Bhoovika/scamsense-xlmroberta` (HF model), `scamsense_pipeline.py` (shared module, see Cell 1a)
**Output**: Logged predictions in Supabase, a tested API, and (optionally) a live Telegram bot


## Cell 1 — Install dependencies

In [1]:
import subprocess, sys, importlib  # subprocess to run pip; importlib to check installed packages

PACKAGES = [  # (import name, pip install spec) for every dependency
    ("numpy",                "numpy<2"),
    ("fastapi",              "fastapi==0.111.0"),
    ("uvicorn",              "uvicorn==0.29.0"),
    ("httpx",                "httpx==0.27.0"),
    ("psycopg2",             "psycopg2-binary==2.9.9"),
    ("sentence_transformers","sentence-transformers==3.0.1"),
    ("langdetect",           "langdetect"),
    ("langgraph",            "langgraph"),
    ("shap",                 "shap"),
    ("pydantic",             "pydantic==2.7.1"),
    ("anyio",                "anyio==4.3.0"),
    ("faiss",                "faiss-cpu"),          
    ("transformers",         "transformers"),        
    ("torch",                "torch"),               
    ("pandas",               "pandas"),              
]

to_install = []  # collects pip specs that still need installing
for import_name, pip_spec in PACKAGES:  # check each package one at a time
    try:
        mod = importlib.import_module(import_name)  # try importing it
        ver = getattr(mod, "__version__", "?")  # read its version if available
        print(f"  already installed  {import_name:<28} {ver}")  # already present, nothing to do
    except ImportError:  # not installed yet
        print(f"   will install       {pip_spec}")  # flag it for installation
        to_install.append(pip_spec)  # queue the pip spec

print(f"\n{len(to_install)} package(s) to install, "
      f"{len(PACKAGES)-len(to_install)} already present.\n")  # summary before installing

if to_install:  # only run pip if something is missing
    print("── Installing ────────────────────────────────────────────")  # section header
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade"] + to_install,
        capture_output=True, text=True
    )  # run pip install for all missing packages at once
    for line in (result.stdout + result.stderr).splitlines():  # scan pip output line by line
        line = line.strip()  # trim whitespace
        if any(kw in line for kw in
               ("Collecting", "Downloading", "Installing", "Successfully",
                "already satisfied", "ERROR", "WARNING", "━")):  # only print noteworthy pip lines
            print(" ", line)  # show the filtered line
    if result.returncode != 0:  # pip failed
        print("\n pip exited with errors — check above")  # flag the failure
    else:
        print("\n All installs done")  # all good
else:
    print(" Nothing to install — all packages already present")  # everything was already installed

  already installed  numpy                        2.0.2
  already installed  fastapi                      0.136.1
  already installed  uvicorn                      0.46.0
  already installed  httpx                        0.28.1
  already installed  psycopg2                     2.9.12 (dt dec pq3 ext lo64)
  already installed  sentence_transformers        5.4.1
   will install       langdetect
  already installed  langgraph                    ?
  already installed  shap                         0.51.0
  already installed  pydantic                     2.12.3
  already installed  anyio                        ?
   will install       faiss-cpu
  already installed  transformers                 5.0.0
  already installed  torch                        2.10.0+cu128
  already installed  pandas                       2.3.3

2 package(s) to install, 13 already present.

── Installing ────────────────────────────────────────────
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.1 MB/s eta 0:0

## Cell 1a — Shared Pipeline Module (`scamsense_pipeline.py`)

Classification, explanation, and risk-scoring logic all live in one shared `scamsense_pipeline.py` module — the identical file used by `04_langgraph_rag.ipynb` — imported here with:

```python
from scamsense_pipeline import init, run_pipeline
```

This replaces the second, independently-maintained copy of the pipeline that used to live in this notebook (which had drifted: only 8 SPF categories vs. NB04's 12, a smaller keyword list, and its own separate SHAP/RAG code).

**Setup (do this once):** upload `scamsense_pipeline.py` as a Kaggle Dataset (or Utility Script) and attach it to this notebook as input. The loader cell below checks a few candidate folders in order and uses whichever one contains the file — see the comments in that cell for the exact paths it checks.


In [2]:
import sys  # to modify the module search path
from pathlib import Path  # for filesystem path handling

# scamsense_pipeline.py lives in exactly one place; every notebook just points
# at it. Adjust the Kaggle dataset slug below if you named it differently.
_CANDIDATE_DIRS = [
    Path.cwd(),                                    # same folder (local / Colab)
    Path("/kaggle/input/datasets/bhoovika/scamsense-utils"),         # Kaggle dataset/utility script
    Path("/kaggle/working"),                       # already copied into working dir
]

for _dir in _CANDIDATE_DIRS:  # check each candidate folder in order
    if (_dir / "scamsense_pipeline.py").exists():  # does the shared module live here?
        sys.path.insert(0, str(_dir))  # make it importable
        break  # stop at the first match found
else:
    raise FileNotFoundError(  # runs only if no candidate matched
        "scamsense_pipeline.py not found in any of: "
        + ", ".join(str(d) for d in _CANDIDATE_DIRS)
        + ". Upload it as a Kaggle dataset/utility script (and add its path to "
        "_CANDIDATE_DIRS above), or place it next to this notebook."
    )

print(f"scamsense_pipeline.py loaded from: {_dir}")  # confirm which path was used


scamsense_pipeline.py loaded from: /kaggle/input/datasets/bhoovika/scamsense-utils


## Cell 2 — Secrets + paths

In [3]:
import os  # (kept for potential OS-level calls; not directly used below)
from pathlib import Path  # filesystem paths
from kaggle_secrets import UserSecretsClient  # Kaggle's secrets API
from urllib.parse import urlparse, urlunparse, quote  # (imported but unused in this notebook)

_s = UserSecretsClient()  # secrets client instance
HF_TOKEN           = _s.get_secret("HF_TOKEN")  # HuggingFace token for the classifier model
GITHUB_TOKEN       = _s.get_secret("GITHUB_TOKEN")  # (loaded but not used — no GitHub push step in this notebook)
SUPABASE_URL       = _s.get_secret("SUPABASE_PROJECT_URL")     # Supabase project REST URL
SUPABASE_ANON_KEY  = _s.get_secret("SUPABASE_ANON_KEY")  # Supabase anon/public API key

# Paths
WORKING_DIR = Path("/kaggle/working")  # Kaggle working directory
OUTPUT_DIR  = WORKING_DIR / "scamsense_output"  # where this notebook writes its outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # create it if missing

print("Secrets loaded")  # confirm secrets were retrieved
print(f"Output dir  : {OUTPUT_DIR}")  # show where outputs will be saved


Secrets loaded
Output dir  : /kaggle/working/scamsense_output


## Cell 3 — Init shared pipeline (model + FAISS index)

Loads the classifier, embedder, and SHAP explainer, and builds/caches the FAISS index from `scamsense_pipeline`'s built-in SPF advisory corpus (no external RAG dataset is passed in here, so it always builds from the module's default corpus rather than loading a separate pre-built index).


In [4]:
from scamsense_pipeline import (  # import shared pipeline functions + constants
    init, classify, explain, get_risk, run_pipeline,
    HF_MODEL_ID, DEVICE,
)

print(f"Device: {DEVICE}")  # confirm CPU/GPU device

# Loads the classifier, embedder, and SHAP explainer, and builds/caches the
# FAISS index from scamsense_pipeline's built-in SPF corpus. No rag_dir is
# passed here, so this always builds fresh rather than loading a pre-built
# Kaggle dataset (unlike NB04, which passes rag_dir=RAG_DIR).
init(hf_token=HF_TOKEN)  # one-time setup of the shared pipeline


Device: cuda
Device: cuda
Loading tokenizer/model from Bhoovika/scamsense-xlmroberta ...


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded
SPF corpus loaded from built-in default (40 chunks)
Building FAISS index from SPF corpus ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index built (40 vectors, dim=384)
SHAP explainer ready
scamsense_pipeline ready


## Cell 4 — Supabase: connect & verify the `predictions` table

This cell does not create the table (assumes it already exists in Supabase); it defines `supabase_insert()` / `supabase_select()` helpers and runs one `supabase_select()` call as a connectivity smoke test.


In [5]:
import httpx  # HTTP client used to call the Supabase REST API
from psycopg2.extras import RealDictCursor  # (imported but unused — REST API is used instead of a direct DB connection)
from datetime import datetime, timezone  # (imported but unused directly here)

SUPABASE_HEADERS = {  # auth + content headers for every Supabase REST call
    "apikey": SUPABASE_ANON_KEY,
    "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=representation"
}

def supabase_insert(record: dict):  # insert one prediction row into the "predictions" table
    r = httpx.post(                              # POST the record to the REST API
        f"{SUPABASE_URL}/rest/v1/predictions",
        headers=SUPABASE_HEADERS,
        json=record,
        timeout=15
    )
    r.raise_for_status()                             # raise an error on a bad HTTP response
    return r.json()                                  # return the inserted row(s)

def supabase_select(query_params: dict = None, limit: int = 20):  # fetch prediction rows, optionally filtered
    params = {"limit": limit, "order": "created_at.desc"}  # default: most recent first, capped at `limit`
    if query_params:  # merge in any extra filters
        params.update(query_params)  # e.g. {"label": "eq.scam"}
    r = httpx.get(                                # GET matching rows from the REST API
        f"{SUPABASE_URL}/rest/v1/predictions",
        headers=SUPABASE_HEADERS,
        params=params,
        timeout=15
    )
    r.raise_for_status()                             # raise an error on a bad HTTP response
    return r.json()                                  # return the matching rows


test = supabase_select(limit=1)  # smoke test: confirm the table is reachable
print("Supabase REST connected")  # confirm connectivity
print("Table 'predictions' ready")  # confirm the table responded

Supabase REST connected
Table 'predictions' ready


## Cell 5 — Presentation helpers + Supabase logging

All detection/explanation/risk logic is now `scamsense_pipeline.run_pipeline()`. This cell only keeps notebook/API-specific concerns: formatting a human-readable explanation string for API responses, and logging predictions to Supabase.

In [6]:
RISK_ADVICE = {  # advisory text shown to the user, keyed by risk tier
    "CRITICAL": "Do NOT transfer any money. Report immediately to SPF at www.police.gov.sg/iwitness or call 1800-255-0000.",
    "HIGH":     "Exercise extreme caution. Verify independently before taking any action. Report suspected scams to SPF.",
    "MEDIUM":   "Be cautious. Do not share personal information or make payments without verification.",
    "LOW":      "Stay alert. If something feels off, trust your instincts and verify through official channels.",
    "NONE":     "This message appears legitimate. Always stay vigilant against scams.",
}

def format_explanation(result: dict) -> str:  # turn a run_pipeline() result into a readable explanation string
    """Build a human-readable explanation string from a scamsense_pipeline.run_pipeline() result."""
    if result["label"] == "ham":  # legitimate message: short reassuring message
        return (
            f"ScamSense classified this message as LEGITIMATE with "
            f"{result['confidence']:.1%} confidence (language: {result['language']}). "
            f"No scam indicators were detected. {RISK_ADVICE['NONE']}"
        )  # build the ham-case explanation text

    feat_str = ""  # optional "key indicators" phrase, built below
    if result.get("top_features"):  # only add this phrase if SHAP tokens exist
        tokens = [f["token"] for f in result["top_features"][:3]]  # take the top 3 SHAP tokens
        feat_str = f" Key scam indicators: {', '.join(tokens)}."  # format them into a sentence fragment

    passages = result.get("rag_chunks") or []  # retrieved SPF advisory passages, if any
    spf_text  = passages[0]["text"] if passages else ""  # text of the top passage
    spf_topic = passages[0]["topic"] if passages else ""  # topic of the top passage
    spf_page  = passages[0].get("source_page", "?") if passages else "?"  # source page of the top passage

    return (
        f"SCAM DETECTED - {result['risk_tier']} risk (score: {result['risk_score']}/100)\n\n"
        f"ScamSense classified this {result['language']}-language message as a "
        f"{result['scam_type']} scam with {result['confidence']:.1%} confidence.{feat_str}\n\n"
        f"SPF Advisory ({spf_topic}, p.{spf_page}):\n"
        f"{spf_text[:300]}...\n\n"
        f"What to do: {RISK_ADVICE[result['risk_tier']]}"
    )  # build the full scam-case explanation text

def log_prediction(record: dict):  # write a prediction record to Supabase, tolerating failures
    try:
        supabase_insert(record)  # attempt the insert
    except Exception as e:  # don't crash the request if logging fails
        print(f"Supabase log failed: {e}")  # just log the error

print("Presentation + logging helpers ready")  # confirm setup


Presentation + logging helpers ready


## Cell 6 — FastAPI app definition

In [7]:
from fastapi import FastAPI, HTTPException  # web framework + HTTP error helper
from fastapi.middleware.cors import CORSMiddleware  # allow cross-origin requests (e.g. from a browser front end)
from pydantic import BaseModel  # request/response schema validation
from typing import Optional  # for optional query parameters
import uvicorn  # ASGI server used to run the app

app = FastAPI(title="ScamSense API", version="1.0.0")  # create the FastAPI application
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])  # allow all origins/methods/headers (demo setting)

# Schemas
class PredictRequest(BaseModel):  # expected JSON body for POST /predict
    message: str  # the text to analyse
    source_app: str = "api"  # caller identifier, defaults to "api"

class StatsResponse(BaseModel):  # expected JSON shape for GET /stats
    total_predictions: int  # total rows logged
    total_scams: int  # rows labelled scam
    total_ham: int  # rows labelled ham
    scam_rate_pct: float  # scam rate as a percentage
    by_risk_tier: dict  # counts grouped by risk tier
    by_language: dict  # counts grouped by language
    by_scam_type: dict  # counts grouped by scam type

# /health
@app.get("/health", tags=["System"])  # register GET /health
def health():  # liveness + DB connectivity check
    try:
        rows = supabase_select(limit=1)
        total = len(rows)
        db_status = "ok"  # quick DB probe
    except Exception as e:
        total = -1
        db_status = f"error: {e}"  # DB unreachable: report the error instead of crashing
    return {
        "status": "ok",
        "model": HF_MODEL_ID,
        "device": DEVICE,
        "db": db_status,
        "total_predictions": total,
    }  # summarise service + DB health

# /predict
@app.post("/predict", tags=["Inference"])  # register POST /predict
def predict(req: PredictRequest):  # run the full pipeline on the submitted message
    result = run_pipeline(req.message)   # <-- shared pipeline, single source of truth
    explanation = format_explanation(result)  # build the human-readable explanation text

    record = {  # assemble the row to store in Supabase
        "message":      result["message"],  # original message
        "language":     result["language"],  # detected language
        "label":        result["label"],  # scam/ham label
        "confidence":   result["confidence"],  # classifier confidence
        "scam_type":    result.get("scam_type"),  # matched SPF taxonomy key
        "risk_score":   result.get("risk_score"),  # numeric risk score
        "risk_tier":    result.get("risk_tier"),  # risk tier
        "explanation":  explanation,  # formatted explanation text
        "sources":      result.get("sources", []),  # SPF advisory source citations
        "top_features": result.get("top_features", []),  # top SHAP tokens
        "source_app":   req.source_app,  # who called the API
    }  # end of record dict
    try:
        inserted = supabase_insert(record)
        result["id"] = inserted[0].get("id") if inserted else None  # log to Supabase and capture the new row id
    except Exception as e:
        print(f"[WARN] DB log failed: {e}")
        result["id"] = None  # don't fail the request if logging fails

    result["explanation"] = explanation  # attach the explanation text to the response
    return result  # send the full result back to the caller

# /stats
@app.get("/stats", tags=["Analytics"], response_model=StatsResponse)  # register GET /stats
def stats():  # aggregate analytics across all logged predictions
    try:
        rows = supabase_select(limit=10000)  # fetch (up to) all logged rows
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))  # surface DB errors as a 500 response

    total = len(rows)                                  # total number of predictions
    scams = sum(1 for r in rows if r.get("label") == "scam")  # count of scam predictions
    ham   = total - scams                              # remaining are ham

    by_tier, by_lang, by_type = {}, {}, {}  # running counters for the breakdowns
    for r in rows:  # tally each row into the breakdowns
        tier  = r.get("risk_tier")  or "NONE"  # default missing tier to NONE
        lang  = r.get("language")   or "unknown"  # default missing language
        stype = r.get("scam_type")  or "unknown"  # default missing scam type
        by_tier[tier]  = by_tier.get(tier, 0)  + 1  # increment tier count
        by_lang[lang]  = by_lang.get(lang, 0)  + 1  # increment language count
        by_type[stype] = by_type.get(stype, 0) + 1  # increment scam-type count

    scam_rate = round(scams / total * 100, 1) if total > 0 else 0.0  # avoid divide-by-zero when there are no rows yet
    return StatsResponse(
        total_predictions=total,
        total_scams=scams,
        total_ham=ham,
        scam_rate_pct=scam_rate,
        by_risk_tier=by_tier,
        by_language=by_lang,
        by_scam_type=by_type,
    )  # package everything into the response schema

# /history
@app.get("/history", tags=["Analytics"])  # register GET /history
def history(limit: int = 20, label: Optional[str] = None):  # return recent predictions, optionally filtered by label
    try:
        params = {"label": f"eq.{label}"} if label else {}
        rows = supabase_select(query_params=params, limit=limit)  # build the optional filter and fetch rows
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    keys = ["id","created_at","message","language","label",
            "confidence","scam_type","risk_tier","risk_score"]  # only return these fields per row
    return {"count": len(rows), "predictions": [{k: r.get(k) for k in keys} for r in rows]}  # trimmed rows + a count

print("FastAPI app defined")  # confirm the app was created
print("Endpoints:")  # list the available routes
print("  GET  /health   - liveness check")
print("  POST /predict  - full pipeline (scamsense_pipeline.run_pipeline) + Supabase log")
print("  GET  /stats    - aggregate analytics")
print("  GET  /history  - recent predictions")


FastAPI app defined
Endpoints:
  GET  /health   - liveness check
  POST /predict  - full pipeline (scamsense_pipeline.run_pipeline) + Supabase log
  GET  /stats    - aggregate analytics
  GET  /history  - recent predictions


## Cell 7 — Run server + test all endpoints

In [8]:
from IPython.display import display  # (imported but unused directly in this cell)
import threading, time, httpx, json, os  # threading to run the server in the background; httpx as the test client

# Kill any existing server on port 8000
os.system("fuser -k 8000/tcp")  # free up port 8000 if a previous run left it bound
time.sleep(1)  # give the OS a moment to release the port

API_BASE = "http://127.0.0.1:8000"                  # base URL of the local server

def _run():                                          # target function for the server thread
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")  # start the FastAPI app

t = threading.Thread(target=_run, daemon=True)        # run the server in a background thread
t.start()                                             # launch it
time.sleep(4)                                         # give it time to come up
print("Server running on", API_BASE)                  # confirm it's up

# NOTE: the block below duplicates the startup above verbatim (starts a
# second server thread on the same port). Left as-is/unchanged; harmless
# since the port is already bound, but kept here only for line-by-line comments.
API_BASE = "http://127.0.0.1:8000"                  # (re-)declare base URL

# Start uvicorn in a background thread
def _run():                                          # target function for the (second) server thread
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")  # start the FastAPI app again

t = threading.Thread(target=_run, daemon=True)        # second background thread
t.start()                                             # launch it
time.sleep(4)                                         # give it time to come up
print("Server running on", API_BASE)                  # confirm (again)

#  Helper 
def pp(label, response):  # pretty-print a labeled API response
    print(f"\n{'='*60}")  # divider line
    print(f"  {label}  [{response.status_code}]")  # show the test label + HTTP status code
    print('='*60)  # divider line
    try:
        data = response.json()  # try to parse + pretty-print JSON
        print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])  # show first 1500 chars of the formatted JSON
    except:  # not JSON: fall back to raw text
        print(response.text[:500])  # show first 500 chars of the raw response

client = httpx.Client(base_url=API_BASE, timeout=120.0)  # reusable HTTP client for all the test calls below

#  Test 1: Health 
pp("GET /health", client.get("/health"))  # check liveness endpoint

#  Test 2: Predict — English phishing 
pp("POST /predict — EN phishing", client.post("/predict", json={
    "message": "Your POSB account has been suspended. Verify at http://posb-secure.xyz",
    "source_app": "notebook_test"
}))  # English phishing example

#  Test 3: Predict — Mandarin impersonation 
pp("POST /predict — ZH impersonation", client.post("/predict", json={
    "message": "您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户。",
    "source_app": "notebook_test"
}))

#  Test 4: Predict — Singlish ecommerce 
pp("POST /predict — Singlish ecommerce", client.post("/predict", json={
    "message": "Selling iPhone 15 Pro Max $400 only lah! PayNow me first then I ship lor.",
    "source_app": "notebook_test"
}))

#  Test 5: Predict — Tamil investment 
pp("POST /predict — TA investment", client.post("/predict", json={
    "message": "உங்கள் முதலீட்டில் 300% லாபம் உறுதி. இப்போதே பதிவு செய்யுங்கள்!",
    "source_app": "notebook_test"
}))

# ── Test 6: Predict — Malay job scam ─────────────────────────────────────────
pp("POST /predict — MS job scam", client.post("/predict", json={
    "message": "Kerja mudah dari rumah! Pendapatan RM5,000 sebulan. Bayar yuran pendaftaran RM200.",
    "source_app": "notebook_test"
}))

#  Test 7: Predict — Ham 
pp("POST /predict — EN ham", client.post("/predict", json={
    "message": "Hi! Your dentist appointment is tomorrow 2pm at Raffles Dental. Reply to confirm.",
    "source_app": "notebook_test"
}))

#  Test 8: Stats 
pp("GET /stats", client.get("/stats"))  # check aggregate analytics

#  Test 9: History 
pp("GET /history?limit=5", client.get("/history?limit=5"))  # check recent-predictions endpoint

client.close()  # release the HTTP client
print("\n All endpoint tests complete")  # confirm all tests ran

Server running on http://127.0.0.1:8000


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


Server running on http://127.0.0.1:8000

  GET /health  [200]
{
  "status": "ok",
  "model": "Bhoovika/scamsense-xlmroberta",
  "device": "cuda",
  "db": "ok",
  "total_predictions": 1
}

  POST /predict — EN phishing  [200]
{
  "message": "Your POSB account has been suspended. Verify at http://posb-secure.xyz",
  "language": "en",
  "label": "scam",
  "confidence": 1.0,
  "scam_prob": 1.0,
  "top_features": [
    {
      "token": "z",
      "shap_value": 0.1627
    },
    {
      "token": "POS",
      "shap_value": 0.16
    },
    {
      "token": "Your ",
      "shap_value": 0.0884
    },
    {
      "token": "suspend",
      "shap_value": 0.0812
    },
    {
      "token": "account ",
      "shap_value": 0.0699
    }
  ],
  "rag_chunks": [
    {
      "id": "spf_019",
      "scam_type": "phishing",
      "topic": "Phishing contact methods",
      "source_page": 11,
      "text": "Phishing scammers contact victims via SMS, email, and messaging platforms with links to fraudulent websi

## Cell 8 — Export prediction log

In [9]:
import subprocess, pandas as pd  # (subprocess unused here); pandas for building the export table

#  Export predictions to CSV via REST 
rows = supabase_select(limit=10000)  # fetch (up to) all logged predictions
df = pd.DataFrame(rows)  # load them into a dataframe


cols = ["id","created_at","message","language","label","confidence",
        "scam_type","risk_score","risk_tier","sources","source_app"]  # columns to keep in the export
df = df[[c for c in cols if c in df.columns]]  # keep only the columns that actually exist

csv_path = OUTPUT_DIR / "05_prediction_log.csv"  # output file path
df.to_csv(csv_path, index=False)  # write the CSV to disk
print(f"Prediction log saved → {csv_path}  ({len(df)} rows)")  # confirm the save
if not df.empty:  # only print a preview if there is data
    print(df[["label","language","risk_tier","scam_type","confidence"]].to_string(index=False))  # preview key columns
    

print("   API endpoints : /health  /predict  /stats  /history")  # recap
print("   DB            : Supabase REST API (persistent)")  # recap
print("   Logged rows   :", len(df))  # recap

Prediction log saved → /kaggle/working/scamsense_output/05_prediction_log.csv  (115 rows)
label language risk_tier                scam_type  confidence
  ham       en      NONE                     None      0.9984
 scam       ms      HIGH                 job_scam      1.0000
 scam       ta  CRITICAL               investment      1.0000
 scam singlish    MEDIUM                ecommerce      1.0000
 scam       zh  CRITICAL government_impersonation      0.9995
 scam       en      HIGH                 phishing      1.0000
 scam       ms      HIGH                 job_scam      1.0000
 scam       ta  CRITICAL               investment      1.0000
  ham       en      NONE                     None      1.0000
 scam       ms      HIGH                 job_scam      1.0000
 scam       ta  CRITICAL               investment      1.0000
 scam       ms      HIGH                 job_scam      1.0000
  ham       en      NONE                     None      0.9984
 scam       ms      HIGH                 j

## Cell 9 — Telegram Bot (ScamSense Scout)

**Prerequisites before running:**
1. Create a bot via [@BotFather](https://t.me/BotFather) → `/newbot` → copy the token.
2. Add the token as a Kaggle secret named `TELEGRAM_BOT_TOKEN`.
3. The FastAPI server from Cell 7 must be running in the background thread.

In [ ]:
# Install python-telegram-bot
import subprocess, sys  # to run pip from within the notebook
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "python-telegram-bot==20.8"],
    capture_output=True, text=True
)  # pin and install python-telegram-bot 20.8
if result.returncode != 0:  # install failed
    print("ERROR:", result.stderr[-400:])  # show the tail of the error output

In [11]:
#  Compatibility patch 
import httpx as _httpx, functools  # needed to monkey-patch AsyncClient below

if not getattr(_httpx.AsyncClient, "_proxies_patched", False):  # only patch once
    _real_init = _httpx.AsyncClient.__init__  # keep a reference to the original __init__
    @functools.wraps(_real_init)  # preserve the original signature/metadata
    def _patched_init(self, *args, **kwargs):  # drop the unsupported "proxies" kwarg before calling init
        kwargs.pop("proxies", None)  # remove it if present
        _real_init(self, *args, **kwargs)  # call the original constructor
    _httpx.AsyncClient.__init__ = _patched_init  # install the patched constructor
    _httpx.AsyncClient._proxies_patched = True  # mark as patched so this only runs once

print("httpx AsyncClient patched   (proxies kwarg suppressed)")  # confirm the patch

import asyncio, httpx, threading, time  # asyncio for the bot event loop; threading to run it in the background
from telegram import Update  # Telegram update object type
from telegram.ext import (
    ApplicationBuilder, CommandHandler, MessageHandler,
    filters, ContextTypes
)  # bot framework building blocks
from kaggle_secrets import UserSecretsClient  # Kaggle's secrets API

#  Load token 
_s = UserSecretsClient()  # secrets client instance
TELEGRAM_TOKEN = _s.get_secret("TELEGRAM_BOT_TOKEN")  # bot token from Kaggle secrets

API_BASE = "http://127.0.0.1:8000"   # FastAPI server from Cell 7

#  Risk emoji map 
TIER_EMOJI = {  # icon shown per risk tier in bot replies
    "CRITICAL": "🔴",
    "HIGH":     "🟠",
    "MEDIUM":   "🟡",
    "LOW":      "🟢",
    "NONE":     "✅",
}

#  /start 
async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):  # handler for the /start command
    await update.message.reply_text(
        "👋 Hi! I'm *ScamSense Scout* — your personal scam checker.\n\n"
        "Just forward me any suspicious message (SMS, email text, WhatsApp) "
        "and I'll analyse it using Singapore Police Force scam intelligence.\n\n"
        "Commands:\n"
        "  /start — show this message\n"
        "  /help  — how to use the bot\n"
        "  /stats — scan statistics",
        parse_mode="Markdown"
    )

#  /help 
async def cmd_help(update: Update, context: ContextTypes.DEFAULT_TYPE):  # handler for the /help command
    await update.message.reply_text(
        "📋 *How to use ScamSense Scout*\n\n"
        "1. Copy or forward any suspicious message to this chat.\n"
        "2. I'll run it through the ScamSense AI pipeline.\n"
        "3. You'll get a risk rating, scam type, and advice.\n\n"
        "Supported languages: English, Singlish, Malay, Tamil, Mandarin.",
        parse_mode="Markdown"
    )

#  /stats 
async def cmd_stats(update: Update, context: ContextTypes.DEFAULT_TYPE):  # handler for the /stats command
    try:
        r = httpx.get(f"{API_BASE}/stats", timeout=15)
        r.raise_for_status()
        d = r.json()  # fetch stats from the FastAPI /stats endpoint
        msg = (
            f"📊 *ScamSense Stats*\n\n"
            f"Total scans : {d['total_predictions']}\n"
            f"Scams found : {d['total_scams']}  ({d['scam_rate_pct']}%)\n"
            f"Legitimate  : {d['total_ham']}\n\n"
            f"*By risk tier:*\n"
        )
        for tier, count in d.get("by_risk_tier", {}).items():  # add one line per risk tier
            emoji = TIER_EMOJI.get(tier, "⚪")  # icon for this tier
            msg += f"  {emoji} {tier}: {count}\n"  # append the tier line
        await update.message.reply_text(msg, parse_mode="Markdown")  # send the stats message
    except Exception as e:  # API call failed
        await update.message.reply_text(f"⚠️ Could not fetch stats: {e}")  # report the error to the user

#  Helper: advice text per tier 
def _advice(tier: str) -> str:  # look up the advisory text for a risk tier
    return {
        "CRITICAL": (
            "Do NOT transfer money. Report immediately to SPF at "
            "www.police.gov.sg/iwitness or call 1800-255-0000."
        ),
        "HIGH": (
            "Exercise extreme caution. Verify independently before "
            "any action. Report to SPF if confirmed."
        ),
        "MEDIUM": (
            "Be cautious. Do not share personal details or make "
            "payments without verifying the sender."
        ),
        "LOW": (
            "Stay alert. If something feels off, trust your instincts "
            "and verify through official channels."
        ),
        "NONE": "This message appears legitimate. Always stay vigilant.",
    }.get(tier, "Stay vigilant and report suspicious messages to SPF.")

#  Message handler (any text -> /predict) 
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):  # handler for any plain text message sent to the bot
    text = update.message.text or update.message.caption or ""  # get the message text (or caption, if it's a media message)
    if not text.strip():                                                       # nothing to analyse
        await update.message.reply_text("Please send a text message to analyse.")  # ask for text
        return                                                                   # stop here

    wait_msg = await update.message.reply_text("🔍 Analysing... please wait.")  # placeholder message while we call the API

    try:
        r = httpx.post(
            f"{API_BASE}/predict",
            json={"message": text, "source_app": "telegram"},
            timeout=120
        )
        r.raise_for_status()
        d = r.json()  # call the FastAPI /predict endpoint
    except Exception as e:                                          # API call failed
        await wait_msg.edit_text(f"❌ Analysis failed: {e}")           # report the failure
        return                                                        # stop here

    label      = d.get("label", "unknown")  # scam/ham label
    conf       = d.get("confidence", 0)  # classifier confidence
    lang       = d.get("language", "?")  # detected language
    scam_type  = d.get("scam_type") or "—"  # matched scam type, if any
    risk_tier  = d.get("risk_tier", "NONE")  # risk tier
    risk_score = d.get("risk_score", 0)  # numeric risk score
    sources    = d.get("sources", [])  # SPF advisory source citations
    emoji      = TIER_EMOJI.get(risk_tier, "⚪")  # icon for this tier

    if label == "ham":  # build a short reassuring reply for legitimate messages
        reply = (
            f"✅ *LEGITIMATE MESSAGE*\n\n"
            f"Confidence : {conf:.1%}\n"
            f"Language   : {lang}\n\n"
            f"No scam indicators detected. Stay vigilant — when in doubt, "
            f"verify through official channels."
        )
    else:
        sources_str = ""  # build a detailed scam-alert reply
        if sources:  # attach SPF sources if any were retrieved
            sources_str = (
                "\n\n📚 *SPF References:*\n"
                + "\n".join(f"• {s}" for s in sources[:2])
            )
        reply = (
            f"{emoji} *SCAM DETECTED — {risk_tier} RISK*\n\n"
            f"Confidence  : {conf:.1%}\n"
            f"Language    : {lang}\n"
            f"Scam type   : {scam_type}\n"
            f"Risk score  : {risk_score}/100"
            f"{sources_str}\n\n"
            f"🛡️ *What to do:*\n{_advice(risk_tier)}"
        )

    print("========== TELEGRAM REPLY ==========")  # also log the reply to the notebook output
    print(reply)  # the reply text
    print("====================================")  # divider
    
    await wait_msg.edit_text(reply)  # replace the "Analysing..." placeholder with the real reply

#  Build and launch the bot 
def run_bot():  # entry point run inside the background thread
    loop = asyncio.new_event_loop()  # each thread needs its own asyncio event loop
    asyncio.set_event_loop(loop)  # make it the active loop for this thread

    app_bot = ApplicationBuilder().token(TELEGRAM_TOKEN).build()  # construct the Telegram bot application
    app_bot.add_handler(CommandHandler("start", cmd_start))  # register /start
    app_bot.add_handler(CommandHandler("help",  cmd_help))  # register /help
    app_bot.add_handler(CommandHandler("stats", cmd_stats))  # register /stats
    app_bot.add_handler(
        MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message)
    )  # route any other text to handle_message

    print("🤖 ScamSense Scout bot starting (long-polling)...")  # confirm startup
    app_bot.run_polling(stop_signals=None)  # start polling Telegram for updates (blocks this thread)

# Daemon thread — cell returns immediately; bot keeps polling in background
bot_thread = threading.Thread(target=run_bot, daemon=True)  # run the bot in a background thread
bot_thread.start()  # launch it

for i in range(10):  # wait up to 10 seconds and confirm the bot is alive
    time.sleep(1)  # check once per second
    if not bot_thread.is_alive():  # thread died early
        print(" Bot thread crashed — check the traceback above.")  # flag the crash
        break  # stop waiting
else:
    print("Bot is live. Open Telegram, find your bot, and send a message.")  # loop completed without crashing: bot is running
    print("   Forward any suspicious text directly to the bot to scan it.")  # usage hint
    print("   To stop: restart the kernel or end the Kaggle session.")  # how to stop it

httpx AsyncClient patched   (proxies kwarg suppressed)
🤖 ScamSense Scout bot starting (long-polling)...
Bot is live. Open Telegram, find your bot, and send a message.
   Forward any suspicious text directly to the bot to scan it.
   To stop: restart the kernel or end the Kaggle session.
========== TELEGRAM REPLY ==========
🔴 *SCAM DETECTED — CRITICAL RISK*

Confidence  : 100.0%
Language    : ta
Scam type   : investment
Risk score  : 95/100

📚 *SPF References:*
• SPF 2025 Annual Scams Brief — Investment scam tactics — platforms (p.18)
• SPF 2025 Annual Scams Brief — Investment scam — fake apps (p.19)

🛡️ *What to do:*
Do NOT transfer money. Report immediately to SPF at www.police.gov.sg/iwitness or call 1800-255-0000.
========== TELEGRAM REPLY ==========
🟠 *SCAM DETECTED — HIGH RISK*

Confidence  : 100.0%
Language    : ms
Scam type   : job_scam
Risk score  : 80/100

📚 *SPF References:*
• SPF 2025 Annual Scams Brief — Job scam statistics 2025 (p.16)
• SPF 2025 Annual Scams Brief — Job sc